#
#
 
F
e
d
e
r
a
t
e
d
 
L
e
a
r
n
i
n
g
 
L
a
b
:
 
L
o
g
i
s
t
i
c
 
r
e
g
r
e
s
s
i
o
n
 
f
o
r
 
c
l
a
s
s
i
f
i
c
a
t
i
o
n
 
o
f
 
h
a
n
d
-
d
r
a
w
n
 
g
e
s
t
u
r
e
s



For this lab, you will train a logistic regression classifier to recognize hand-drawn gestures from the Quick Draw dataset. Then you will test it on real hand drawings submitted by your classmates.

The lab has four setups:
- **Setup 0:** Train on Quick Draw data, test on Quick Draw
- **Setup 1:** Same model, tested on 60 hand-drawn images from 6 classmates
- **Setup 2:** Federated Learning with Student A (first 3 people, 30 images)
- **Setup 3:** FL with both students (A: 3 people, B: 3 people)


#### Setup 0 — Train a classifier on Quick Draw data

First, we'll train a logistic regression on a subset of the Quick Draw dataset. 200 samples per class for your student slice.


In [ ]:
!pip install flwr scikit-learn matplotlib numpy Pillow nest-asyncio seaborn --quiet


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import urllib.request
import warnings
from PIL import Image
import PIL.ImageOps
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import log_loss
import flwr as fl

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

print('All packages imported.')


In [ ]:
STUDENT_ID = 'A'                # 'A' or 'B'
SERVER_ADDRESS = 'localhost:8090'

CLASSES = ['cat', 'dog', 'sun', 'clock', 'mountain',
           'tent', 'tree', 'bird', 'star', 'face']
nclasses = len(CLASSES)
SAMPLES = 200
SLICE_START = {'A': 0, 'B': 200}[STUDENT_ID]
SERVER_IP = SERVER_ADDRESS.split(':')[0]
HTTP_PORT = 8081

print(f'Student: {STUDENT_ID}  |  Server: {SERVER_ADDRESS}')


In [ ]:
# Load Quick Draw data for this student (200 samples per class)
BASE = 'https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/'
X, y = [], []
for i, c in enumerate(['cat','dog','sun','clock','mountain',
                        'tent','tree','bird','star','face']):
    fn = c + '.npy'
    if not os.path.exists(fn): urllib.request.urlretrieve(BASE + fn, fn)
    X.append(np.load(fn)[SLICE_START:SLICE_START+SAMPLES].copy().astype('f4')/255.)
    y.append(np.full(SAMPLES, i))
X = np.vstack(X); y = np.concatenate(y)
print(f'Loaded {len(X)} samples ({SAMPLES} per class)')


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=9,
                                     train_size=0.7, test_size=0.3)


In [ ]:
clf = LogisticRegression(penalty='l2',
                         max_iter=1000, solver='saga').fit(X_train, y_train)


In [ ]:
accuracy = clf.score(X_test, y_test)
print(accuracy)


In [ ]:
y_pred_qd = clf.predict(X_test)
cm = confusion_matrix(y_test, y_pred_qd)

fig, ax = plt.subplots(figsize=(10, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Setup 0 — Confusion Matrix (Quick Draw test set)')
plt.xticks(rotation=45, ha='right')
plt.show()


In [ ]:
misclassified_qd = np.where(y_test != y_pred_qd)[0]

plt.figure(figsize=(15, 6))
plt.suptitle('Misclassified Examples (Setup 0 — Quick Draw)', fontsize=16)

for i, idx in enumerate(misclassified_qd[:10]):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[idx].reshape(28, 28), cmap=plt.cm.gray)
    plt.title(f'True: {CLASSES[y_test[idx]]}\nPred: {CLASSES[y_pred_qd[idx]]}', color='red')
    plt.axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


#### Upload hand-drawn images (60 individual photos)

Organize your files as `INITIALS_CLASS.jpg` (e.g., `SD_cat.jpg`, `JD_dog.jpg`), then zip them into one file and upload below.


In [ ]:
from google.colab import files
import zipfile, io

uploaded = files.upload()
zip_fn = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_fn, 'r') as zf:
    zf.extractall('hand_drawings')

print(f'Extracted {zip_fn} to hand_drawings/')
!ls hand_drawings/ | head -10


In [ ]:
# Batch preprocess all 60 images
# Filenames must follow INITIALS_CLASS.jpg format
import os, re

X_hand, y_hand, persons = [], [], []

# Edit rotation per person (0, 90, 180, 270)
PERSON_ROTATIONS = {
    'SD': 0,  # change these
}

for fn in sorted(os.listdir('hand_drawings/')):
    if not fn.lower().endswith(('.jpg','.jpeg','.png')):
        continue
    match = re.match(r'([A-Z]+)_([a-z]+)\.', fn, re.I)
    if not match:
        print(f'Skipping {fn}: does not match INITIALS_CLASS pattern')
        continue
    initials = match.group(1).upper()
    label_str = match.group(2).lower()
    if label_str not in CLASSES:
        print(f'Skipping {fn}: unknown class "{label_str}"')
        continue
    label = CLASSES.index(label_str)

    img = Image.open(os.path.join('hand_drawings/', fn)).convert('L')
    rot = PERSON_ROTATIONS.get(initials, 0)
    if rot:
        img = img.rotate(-rot, expand=True, fillcolor='white')
    img = img.resize((28, 28), Image.LANCZOS)
    img_arr = np.array(img, dtype=np.float32)
    img_arr = (255 - img_arr) / 255.0

    X_hand.append(img_arr.reshape(784))
    y_hand.append(label)
    persons.append(initials)

X_hand = np.array(X_hand); y_hand = np.array(y_hand)
persons = np.array(persons)
print(f'Loaded {len(X_hand)} images from {len(np.unique(persons))} people: {np.unique(persons)}')


In [ ]:
# Verify: each row = class, each column = person
n_people = len(np.unique(persons))
fig, axes = plt.subplots(10, n_people, figsize=(14, 20))
for c in range(10):
    idxs = np.where(y_hand == c)[0]
    for j, p in enumerate(np.unique(persons)):
        p_idx = idxs[persons[idxs] == p]
        if len(p_idx):
            axes[c, j].imshow(X_hand[p_idx[0]].reshape(28, 28), cmap='gray')
        axes[c, j].set_title(p, fontsize=8)
        axes[c, j].axis('off')
    axes[c, 0].set_ylabel(CLASSES[c], fontsize=14)
plt.tight_layout()
plt.show()


#### Setup 1 — Test your Quick Draw model on real hand drawings

Now test the Setup 0 model on the 60 hand-drawn images you just processed. This shows how well a model trained on Quick Draw (computer-drawn strokes) generalizes to real hand drawings.


In [ ]:
setup1_pred = clf.predict(X_hand)
setup1_acc = np.mean(setup1_pred == y_hand)
print(f'Setup 1 accuracy on hand-drawn images: {setup1_acc:.1%}')


In [ ]:
cm = confusion_matrix(y_hand, setup1_pred)
fig, ax = plt.subplots(figsize=(10, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Setup 1 — Confusion Matrix (hand-drawn images)')
plt.xticks(rotation=45, ha='right')
plt.show()


In [ ]:
misclassified = np.where(y_hand != setup1_pred)[0]
print(f'Misclassified {len(misclassified)} / {len(y_hand)}')
plt.figure(figsize=(15, 8))
plt.suptitle('Misclassified Examples (Setup 1 — hand-drawn)', fontsize=16)
n_show = min(10, len(misclassified))
for i in range(n_show):
    idx = misclassified[i]
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_hand[idx].reshape(28, 28), cmap=plt.cm.gray)
    plt.title(f'True: {CLASSES[y_hand[idx]]} ({persons[idx]})\nPred: {CLASSES[setup1_pred[idx]]}',
              color='red', fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()


#### How the Grid Extraction Works

The 60 individual photos are processed from a single zip file:

1. **Upload** — one zip file containing all photos named `INITIALS_CLASS.jpg`
2. **Filename parsing** — `SD_cat.jpg` is split into initials=`SD`, class=`cat`
3. **Batch preprocess** — each photo is grayscaled, resized to 28x28, inverted (white strokes on black), and scaled to [0,1]
4. **Storage** — all 60 images stored in `X_hand` with labels in `y_hand` and person initials in `persons`

OpenCV contour detection `->` perspective warp `->` uniform grid slice `->` `Image.resize(28,28)` `->` `(255 - cell) / 255.0`


#### Why Setup 1 Has Low Accuracy

The model trained in Setup 0 learned patterns from **Quick Draw** — clean, perfectly centered, computer-generated strokes with consistent line width and minimal noise.

Real hand drawings are different: uneven pressure, varying stroke width, positioning differences, and background noise. The model has never seen these variations, so it performs at **chance level (about 10% for 10 classes)** — it is essentially guessing.

This is expected and intentional. It demonstrates that *a model trained on clean digital strokes does not generalize to messy hand drawings.* The confusion matrix and misclassified examples show what the model finds confusing.


#### How Federated Learning Fixes This

Federated Learning (Setups 2 and 3) retrains the model on **your actual hand-drawn images** instead of Quick Draw data:

- **Setup 2** — Student A participates with 3 people (30 images). The model adapts to As drawing style.
- **Setup 3** — Both students participate (A: 30 images, B: 30 images). More diverse data = better generalization.

Each FL round:
1. Server sends the current global model to clients
2. Each client trains for 1 epoch on their local hand-drawn data
3. Clients send updated weights back to the server
4. Server averages the weights (FedAvg) and saves the aggregated model

The server uses Flower's FedAvg strategy with SaveOnAggregateStrategy and serves model files over HTTP.


#### Federated Learning

Now you will run Federated Learning (FL) rounds using the hand-drawn images as training data instead of Quick Draw. Your raw data never leaves your computer.

- **Setup 2:** Student A participates with 3 people (30 images)
- **Setup 3:** Both students participate (A: 3 people, B: 3 people)


In [ ]:
class SketchClient(fl.client.NumPyClient):
    def __init__(self, X_train, y_train, X_val, y_val):
        self.X_train = X_train; self.y_train = y_train
        self.X_val = X_val; self.y_val = y_val
        self.model = LogisticRegression(
            penalty='l2', max_iter=1, warm_start=True, solver='saga',
        )
        dummy_X = np.zeros((nclasses * 2, 784))
        dummy_y = np.repeat(np.arange(nclasses), 2)
        self.model.fit(dummy_X, dummy_y)
    def get_parameters(self, config):
        return [self.model.coef_, self.model.intercept_]
    def fit(self, parameters, config):
        self.model.coef_ = parameters[0]
        self.model.intercept_ = parameters[1]
        self.model.fit(self.X_train, self.y_train)
        return [self.model.coef_, self.model.intercept_], len(self.X_train), {}
    def evaluate(self, parameters, config):
        self.model.coef_ = parameters[0]
        self.model.intercept_ = parameters[1]
        loss = float(log_loss(self.y_val, self.model.predict_proba(self.X_val)))
        acc = float(self.model.score(self.X_val, self.y_val))
        return loss, len(self.X_val), {'accuracy': acc}
print('SketchClient ready.')


#### Setup 2 — Federated Learning (Student A, first 3 people)

Student A trains on images from the first 3 people via FL. After rounds complete, the aggregated model is tested on all 60 hand-drawn images.


In [ ]:
# Split people into Student A (first 3) and Student B (last 3)
unique_people = sorted(np.unique(persons))
a_people = unique_people[:3]
b_people = unique_people[3:]
X_fl_a = X_hand[np.isin(persons, a_people)]
y_fl_a = y_hand[np.isin(persons, a_people)]
X_fl_b = X_hand[np.isin(persons, b_people)]
y_fl_b = y_hand[np.isin(persons, b_people)]
print(f"Student A: {len(X_fl_a)} images (people: {a_people})")
print(f"Student B: {len(X_fl_b)} images (people: {b_people})")


In [ ]:
print('Connecting to', SERVER_ADDRESS, 'for Setup 2 (A only, 3 people)...')
client = SketchClient(X_fl_a, y_fl_a, X_test, y_test)
fl.client.start_numpy_client(server_address=SERVER_ADDRESS, client=client)
fl_model_s2 = client.model
print('Setup 2 complete.')


In [ ]:
# Evaluate Setup 2 model on all 60 hand-drawn images
s2_pred = fl_model_s2.predict(X_hand)
s2_acc = np.mean(s2_pred == y_hand)
print(f'Setup 2 accuracy: {s2_acc:.1%}')

cm = confusion_matrix(y_hand, s2_pred)
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES).plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Setup 2 — Confusion Matrix (hand-drawn images)')
plt.xticks(rotation=45, ha='right')
plt.show()

mis = np.where(y_hand != s2_pred)[0]
print(f'Misclassified {len(mis)}/{len(y_hand)}')
plt.figure(figsize=(15, 8))
plt.suptitle('Setup 2 — Misclassified (hand-drawn)', fontsize=16)
for i, idx in enumerate(mis[:10]):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_hand[idx].reshape(28, 28), cmap=plt.cm.gray)
    plt.title(f'True: {CLASSES[y_hand[idx]]} ({persons[idx]})\nPred: {CLASSES[s2_pred[idx]]}',
              color='red', fontsize=9)
    plt.axis('off')
plt.tight_layout(); plt.show()


#### Setup 3 — Federated Learning (Both students, A: 3 + B: 3 people)

Both students participate. A trains on 3 people (30 images), B on 3 people (30 images). Server waits for both clients before each round.


In [ ]:
print('Connecting to', SERVER_ADDRESS, 'for Setup 3...')
my_sheets = X_fl_a if STUDENT_ID == 'A' else X_fl_b
my_labels = y_fl_a if STUDENT_ID == 'A' else y_fl_b
client = SketchClient(my_sheets, my_labels, X_test, y_test)
fl.client.start_numpy_client(server_address=SERVER_ADDRESS, client=client)
fl_model_s3 = client.model
print('Setup 3 complete.')


In [ ]:
# Evaluate Setup 3 model on all 60 hand-drawn images
s3_pred = fl_model_s3.predict(X_hand)
s3_acc = np.mean(s3_pred == y_hand)
print(f'Setup 3 accuracy: {s3_acc:.1%}')

cm = confusion_matrix(y_hand, s3_pred)
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES).plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Setup 3 — Confusion Matrix (hand-drawn images)')
plt.xticks(rotation=45, ha='right')
plt.show()

mis = np.where(y_hand != s3_pred)[0]
print(f'Misclassified {len(mis)}/{len(y_hand)}')
plt.figure(figsize=(15, 8))
plt.suptitle('Setup 3 — Misclassified (hand-drawn)', fontsize=16)
for i, idx in enumerate(mis[:10]):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_hand[idx].reshape(28, 28), cmap=plt.cm.gray)
    plt.title(f'True: {CLASSES[y_hand[idx]]} ({persons[idx]})\nPred: {CLASSES[s3_pred[idx]]}',
              color='red', fontsize=9)
    plt.axis('off')
plt.tight_layout(); plt.show()


#### Compare all models on the hand-drawn test set


In [ ]:
models = [
    ('Setup 0 (Quick Draw)', clf, setup1_acc),
    ('Setup 2 (FL, A only)', fl_model_s2, s2_acc),
    ('Setup 3 (FL, both)', fl_model_s3, s3_acc),
]
print(f'{"Model":<30} {"Accuracy":<12}  Prediction breakdown by person')
print('-' * 80)
for name, m, acc in models:
    preds = m.predict(X_hand)
    line = f'{name:<30} {acc:<8.1%}  '
    for p in np.unique(persons):
        mask = persons == p
        p_acc = np.mean(preds[mask] == y_hand[mask])
        line += f'{p}:{p_acc:.0%} '
    print(line)


#### Done!

| Concept | What happened |
|---------|--------------|
| **Setup 0** | Trained on Quick Draw, tested on Quick Draw |
| **Setup 1** | Same model tested on real hand-drawn images |
| **Setup 2** | FL with Student A's hand-drawn data (3 people) |
| **Setup 3** | FL with both students (A: 3, B: 3 people) |
| **Privacy** | Raw data never left each student's computer |
